<a href="https://colab.research.google.com/github/Drewbits/petrophysical-data-quality-workflow/blob/main/03_Build_DLIS_Metadata_and_Curve_Catalog.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Business Objective: The goal is to move from one DLIS file to a metadata catalog across all 34 DLIS files.

In [1]:
!pip install -q dlisio

In [2]:
from google.colab import drive
from pathlib import Path

import pandas as pd
import numpy as np

from dlisio import dlis

drive.mount("/content/drive")

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

print("Environment ready.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Environment ready.


In [3]:
DATASET_ROOT = Path("/content/drive/MyDrive/Datasets/15_9-19 A/04.COMPOSITE")

print(f"Dataset exists: {DATASET_ROOT.exists()}")

Dataset exists: True


In [4]:
dlis_files = sorted(
    path
    for path in DATASET_ROOT.rglob("*")
    if path.is_file() and path.suffix.upper() == ".DLIS"
)

print(f"DLIS files found: {len(dlis_files)}")

for path in dlis_files[:10]:
    print(path.relative_to(DATASET_ROOT))

DLIS files found: 34
15_9-F-1/WLC_COMPOSITE_1.DLIS
15_9-F-1/WLC_PETROPHYSICAL_COMPOSITE_1.DLIS
15_9-F-1 A/WLC_COMPOSITE_1.DLIS
15_9-F-1 A/WLC_PETROPHYSICAL_COMPOSITE_1.DLIS
15_9-F-1 B/WLC_COMPOSITE_1.DLIS
15_9-F-1 B/WLC_PETROPHYSICAL_COMPOSITE_1.DLIS
15_9-F-1 C/WLC_COMPOSITE_1.DLIS
15_9-F-1 C/WLC_PETROPHYSICAL_COMPOSITE_1.DLIS
15_9-F-10/WLC_COMPOSITE_1.DLIS
15_9-F-10/WLC_PETROPHYSICAL_COMPOSITE_1.DLIS


In [5]:
test_path = dlis_files[0]

print("File name:", test_path.name)
print("Well name:", test_path.parent.name)

File name: WLC_COMPOSITE_1.DLIS
Well name: 15_9-F-1


In [6]:
DATASET_ROOT = Path("/content/drive/MyDrive/Datasets/15_9-19 A/04.COMPOSITE")

Proceed with the catalog build.

Step 1 — initialize the containers

In [7]:
catalog_records = []
load_errors = []

Step 2 — loop through all DLIS files

In [8]:
for file_path in dlis_files:

    well_name = file_path.parent.name

    try:
        physical_file = dlis.load(file_path)

        for logical_index, logical_file in enumerate(physical_file):

            for frame_index, frame in enumerate(logical_file.frames):

                for channel in frame.channels:

                    catalog_records.append(
                        {
                            "well_name": well_name,
                            "file_name": file_path.name,
                            "relative_path": str(
                                file_path.relative_to(DATASET_ROOT)
                            ),
                            "logical_file_index": logical_index,
                            "frame_index": frame_index,
                            "frame_name": frame.name,
                            "index_type": frame.index_type,
                            "mnemonic": channel.name,
                            "long_name": channel.long_name,
                            "units": channel.units,
                        }
                    )

    except Exception as error:

        load_errors.append(
            {
                "well_name": well_name,
                "file_name": file_path.name,
                "relative_path": str(
                    file_path.relative_to(DATASET_ROOT)
                ),
                "error_type": type(error).__name__,
                "error_message": str(error),
            }
        )

Instead of: Open one file,
Inspect one frame,
Inspect one set of channels

34 DLIS files
    ↓

Every logical file
    ↓

Every frame
    ↓

Every channel
    ↓

One centralized catalog

Step 3 — turn the results into DataFrames

In [9]:
catalog_df = pd.DataFrame(catalog_records)
errors_df = pd.DataFrame(load_errors)

print(f"Catalog rows created: {len(catalog_df):,}")
print(f"Files with loading errors: {len(errors_df)}")

Catalog rows created: 1,078
Files with loading errors: 0


In [10]:
catalog_df.head(20)

,well_name,file_name,relative_path,logical_file_index,frame_index,frame_name,index_type,mnemonic,long_name,units
0,15_9-F-1,WLC_COMPOSITE_1.DLIS,15_9-F-1/WLC_COMPOSITE_1.DLIS,0,0,0,BOREHOLE-DEPTH,DEPTH,,mm
1,15_9-F-1,WLC_COMPOSITE_1.DLIS,15_9-F-1/WLC_COMPOSITE_1.DLIS,0,0,0,BOREHOLE-DEPTH,GR,Gamma Ray,gAPI
2,15_9-F-1,WLC_COMPOSITE_1.DLIS,15_9-F-1/WLC_COMPOSITE_1.DLIS,0,0,0,BOREHOLE-DEPTH,CALI,Caliper,in
3,15_9-F-1,WLC_COMPOSITE_1.DLIS,15_9-F-1/WLC_COMPOSITE_1.DLIS,0,0,0,BOREHOLE-DEPTH,RDEP,Deep Resistivity,ohm.m
4,15_9-F-1,WLC_COMPOSITE_1.DLIS,15_9-F-1/WLC_COMPOSITE_1.DLIS,0,0,0,BOREHOLE-DEPTH,RMED,Medium Resistivity,ohm.m
5,15_9-F-1,WLC_COMPOSITE_1.DLIS,15_9-F-1/WLC_COMPOSITE_1.DLIS,0,0,0,BOREHOLE-DEPTH,DEN,Density,g/cm3
6,15_9-F-1,WLC_COMPOSITE_1.DLIS,15_9-F-1/WLC_COMPOSITE_1.DLIS,0,0,0,BOREHOLE-DEPTH,DENC,Density Correction,g/cm3
7,15_9-F-1,WLC_COMPOSITE_1.DLIS,15_9-F-1/WLC_COMPOSITE_1.DLIS,0,0,0,BOREHOLE-DEPTH,PEF,Photoelectric Factor,b/e
8,15_9-F-1,WLC_COMPOSITE_1.DLIS,15_9-F-1/WLC_COMPOSITE_1.DLIS,0,0,0,BOREHOLE-DEPTH,NEU,Neutron,v/v
9,15_9-F-1,WLC_COMPOSITE_1.DLIS,15_9-F-1/WLC_COMPOSITE_1.DLIS,0,0,0,BOREHOLE-DEPTH,AC,Sonic Compressional,us/ft


Step 4 — inspect the shape of the catalog

In [11]:
print(f"Catalog shape: {catalog_df.shape}")

print("\nColumns:")
print(catalog_df.columns.tolist())

Catalog shape: (1078, 10)

Columns:
['well_name', 'file_name', 'relative_path', 'logical_file_index', 'frame_index', 'frame_name', 'index_type', 'mnemonic', 'long_name', 'units']


Step 5 — count the wells represented

In [12]:
unique_wells = sorted(catalog_df["well_name"].unique())

print(f"Unique wells represented: {len(unique_wells)}")

for well in unique_wells:
    print(well)

Unique wells represented: 17
15_9-F-1
15_9-F-1 A
15_9-F-1 B
15_9-F-1 C
15_9-F-10
15_9-F-11
15_9-F-11 A
15_9-F-11 B
15_9-F-11 T2
15_9-F-15
15_9-F-15 A
15_9-F-15 B
15_9-F-15 C
15_9-F-15 D
15_9-F-5
15_9-F-9
15_9-F-9 A


Your batch process successfully handled:

34 DLIS files
17 wells
1,078 channel-level metadata records
0 loading failures

That is a meaningful result. You have now moved from single-file inspection to archive-level metadata extraction.

The next step is to profile the catalog itself.

In [13]:
unique_mnemonics = sorted(catalog_df["mnemonic"].dropna().unique())

print(f"Unique mnemonics: {len(unique_mnemonics)}")

for mnemonic in unique_mnemonics:
    print(mnemonic)

Unique mnemonics: 229
A16H
A22H
A28H
A34H
A40H
ABDC01M
ABDC02M
ABDC03M
ABDC04M
ABDC05M
ABDC06M
ABDC07M
ABDC08M
ABDC09M
ABDC10M
ABDC11M
ABDC12M
ABDC13M
ABDC14M
ABDC15M
ABDC16M
ABDCQF01
ABDCQF02
ABDCQF03
ABDCQF04
AC
ACS
APRA01M
APRA02M
APRA03M
APRA04M
APRA05M
APRA06M
APRA07M
APRA08M
APRA09M
APRA10M
APRA11M
APRA12M
APRA13M
APRA14M
APRA15M
APRA16M
ARTM
ATMP
ATMP_MWD
AZRIT1T2
AZRTBM
BDCFM
BDSIM
BPHI
BS
BVILINE
CALCM
CALI
CBWLINE
CHRP
CHTP
CRPM
DCAV
DCHO
DCVE
DEN
DENC
DEPT
DEPTH
DPEFM
DRHB
DRHFM
DRHL
DRHO
DRHR
DRHU
DTC
DTCO
DTHM
DTRP
DTRS
DTS
DTSM
DTTP
DTTS
DWAL_WAL
DWAL_WALK2
DWCA_WAL
DWCA_WALK2
DWFE_WAL
DWFE_WALK2
DWGD_WAL
DWGD_WALK2
DWSI_WAL
DWSI_WALK2
DWSU_WAL
DWSU_WALK2
DWTI_WAL
DWTI_WALK2
DXFE_WAL
DXFE_WALK2
GR
GRAFM
GRCDFM
GRCFM
GRCLFM
GRCRFM
GRCS01M
GRCS02M
GRCS03M
GRCS04M
GRCS05M
GRCS06M
GRCS07M
GRCS08M
GRCUFM
GRM1
GRMA
GRSIM
GR_ARC
GR_ARC_FILT
MBVI
MBVM
MBW
MCBW
MPERM
MPHE
MPHS
MWD_GR_BHC
NBGRCFM
NBGRCS01
NBGRCS02
NBGRCS03
NBGRCS04
NBGRCS05
NBGRCS06
NBGRCS07
NBGRCS08
NBGRCS09
NBGRC

This begins answering three important data-management questions:

Which curves are common across the archive?

Which curves only occur in a few wells?

Does the same mnemonic appear with different units?

In [14]:
unit_variation = (
    catalog_df.groupby("mnemonic")["units"]
    .nunique()
    .sort_values(ascending=False)
)

unit_variation[unit_variation > 1]

,units
mnemonic,
AC,3
RMED,3
RDEP,3
A40H,2
A34H,2
DTRP,2
DTCO,2
DWAL_WAL,2
DTTP,2


If that returns anything, we have found mnemonics that are associated with more than one unit somewhere in the archive.

In [15]:
well_curve_counts = (
    catalog_df.groupby("well_name")["mnemonic"]
    .nunique()
    .sort_values(ascending=False)
)

well_curve_counts

,mnemonic
well_name,
15_9-F-11 T2,98
15_9-F-1,90
15_9-F-15 D,89
15_9-F-15,71
15_9-F-15 C,71
15_9-F-1 C,64
15_9-F-10,64
15_9-F-11 B,63
15_9-F-15 B,61


In [16]:
catalog_df["composite_type"] = np.where(
    catalog_df["file_name"].str.contains(
        "PETROPHYSICAL",
        case=False,
        na=False
    ),
    "Petrophysical Composite",
    "Composite"
)

catalog_df["composite_type"].value_counts()

,count
composite_type,
Petrophysical Composite,890
Composite,188


Your archive contains 229 distinct mnemonics across 17 wells, even though our first simple composite contained only 12 channels. That tells us the petrophysical composites contain a much richer and less standardized set of measurements.

More importantly, many mnemonics occur with multiple units. Some have three:

AC      3 units

RDEP    3 units

RMED    3 units

and many others have two, including:

DEPTH

GR

CALI

DEN

DENC

PEF

NEU

ROP

BS

We should not conclude yet that these are errors. There are several possibilities:

equivalent units written differently (g/cm3 vs G/C3);
genuinely different units requiring conversion;
blank/missing unit metadata;
different definitions being assigned the same mnemonic;
vendor-specific conventions.

Our job now is to determine which.

Also, the archive is heavily weighted toward the richer products:

890 / 1,078 = ~82.6% of catalog records come from Petrophysical Composite files.

And curve availability varies dramatically—from 17 unique mnemonics in 15_9-F-11 to 98 in 15_9-F-11 T2.

That variability is precisely why a metadata catalog is valuable.

Next: inspect the actual unit combinations

In [17]:
unit_details = (
    catalog_df.groupby("mnemonic")["units"]
    .apply(lambda x: sorted(set(x.dropna())))
    .to_frame("unit_variants")
)

unit_details["unit_count"] = unit_details["unit_variants"].apply(len)

unit_details = unit_details[
    unit_details["unit_count"] > 1
].sort_values("unit_count", ascending=False)

unit_details

,unit_variants,unit_count
mnemonic,,
AC,"[US/F, US/FT, us/ft]",3
RDEP,"[OHM.M, OHMM, ohm.m]",3
RMED,"[OHM.M, OHMM, ohm.m]",3
A40H,"[OHMM, ohm.m]",2
A34H,"[OHMM, ohm.m]",2
ATMP,"[DEGC, degC]",2
BS,"[IN, in]",2
CALI,"[IN, in]",2
ACS,"[US/F, us/ft]",2


Now let's investigate several important petrophysical curves individually:

In [18]:
curves_to_inspect = [
    "DEPTH",
    "GR",
    "CALI",
    "RDEP",
    "RMED",
    "DEN",
    "DENC",
    "PEF",
    "NEU",
    "AC",
    "ROP",
    "BS",
]

unit_comparison = (
    catalog_df[
        catalog_df["mnemonic"].isin(curves_to_inspect)
    ][
        [
            "well_name",
            "file_name",
            "mnemonic",
            "long_name",
            "units",
        ]
    ]
    .drop_duplicates()
    .sort_values(["mnemonic", "units", "well_name"])
)

unit_comparison

,well_name,file_name,mnemonic,long_name,units
312,15_9-F-10,WLC_COMPOSITE_1.DLIS,AC,Delta-T Compressional,US/F
625,15_9-F-15,WLC_COMPOSITE_2.DLIS,AC,Delta-T Compressional,US/F
698,15_9-F-15 A,WLC_COMPOSITE_2.DLIS,AC,Delta-T Compressional,US/F
756,15_9-F-15 B,WLC_COMPOSITE_2.DLIS,AC,Delta-T Compressional,US/F
818,15_9-F-15 C,WLC_COMPOSITE_2.DLIS,AC,Delta-T Compressional,US/F
...,...,...,...,...,...
373,15_9-F-11,WLC_COMPOSITE_1.DLIS,ROP,Rate of Penetration,m/h
398,15_9-F-11 A,WLC_COMPOSITE_1.DLIS,ROP,Rate of Penetration,m/h
457,15_9-F-11 B,WLC_COMPOSITE_1.DLIS,ROP,Rate of Penetration,m/h
523,15_9-F-11 T2,WLC_COMPOSITE_1.DLIS,ROP,Rate of Penetration,m/h


Why we're doing this

We're moving from:

"AC has three units."

to:

"What are those three units, which wells use them, and do they actually represent the same measurement?"

That distinction is fundamental to normalization.

For example, suppose we eventually discover:

AC

├── us/ft

├── us/m

└── µs/ft

Two may simply be alternate representations of the same unit, while us/m would require a numerical conversion before combining the data.

Similarly, if:

DEN

├── g/cm3

└── kg/m3

We would need both mnemonic standardization and unit conversion.

That's the beginning of the ETL/data-governance layer of this project:

Raw DLIS metadata
        ↓

Metadata catalog
        ↓

Identify inconsistencies       ← WE ARE HERE
        ↓

Canonical mnemonic mapping
        ↓

Canonical unit mapping
        ↓

Unit conversion
        ↓

Validated standardized data
        ↓
        
SQL database

This is a useful result. Most of the “unit inconsistency” is actually formatting inconsistency, not true dimensional inconsistency.

For example:

GR: GAPI vs gAPI → same unit

DEN: G/CC vs g/cm3 → same physical unit

NEU: V/V vs v/v → same unit

RDEP / RMED: OHM.M, OHMM, ohm.m → same resistivity unit

AC: US/F, US/FT, us/ft → same slowness unit

BS, CALI: IN vs in → same unit

ROP: M/H vs m/h → same unit

So the first standardization task should be unit canonicalization, not numerical conversion.

There are, however, a few entries that deserve closer inspection before we automate anything:

DEPTH: [0.1 in, mm]

TNPH: [PU, V/V]
several mineral/weight-fraction curves with KGF/ vs KGF/KGF

blank units vs unitless

Those may represent true scaling or metadata issues.

Next step: classify the unit variants

In [19]:
unit_mapping = {
    "GAPI": "gAPI",
    "gAPI": "gAPI",

    "OHM.M": "ohm.m",
    "OHMM": "ohm.m",
    "ohm.m": "ohm.m",

    "US/F": "us/ft",
    "US/FT": "us/ft",
    "us/ft": "us/ft",

    "IN": "in",
    "in": "in",

    "G/CC": "g/cm3",
    "g/cm3": "g/cm3",

    "V/V": "v/v",
    "v/v": "v/v",

    "B/E": "b/e",
    "b/e": "b/e",

    "M/H": "m/h",
    "m/h": "m/h",

    "DEGC": "degC",
    "degC": "degC",

    "S": "s",
    "s": "s",

    "unitless": "unitless",
    "": "unknown",
}

In [20]:
catalog_df["standard_unit"] = (
    catalog_df["units"]
    .fillna("")
    .map(unit_mapping)
    .fillna(catalog_df["units"])
)

In [21]:
standardized_unit_variation = (
    catalog_df.groupby("mnemonic")["standard_unit"]
    .nunique()
    .sort_values(ascending=False)
)

standardized_unit_variation[
    standardized_unit_variation > 1
]

,standard_unit
mnemonic,
DXFE_WAL,2
DWTI_WAL,2
DWSU_WAL,2
CRPM,2
DEPTH,2
DWAL_WAL,2
DWCA_WAL,2
DWSI_WAL,2
DWFE_WAL,2


We now have 20 mnemonics that still need investigation

Let's inspect exactly what remains.

Step 1 — Build a detailed conflict table

In [22]:
remaining_conflicts = (
    catalog_df[
        catalog_df["mnemonic"].isin(
            standardized_unit_variation[
                standardized_unit_variation > 1
            ].index
        )
    ][
        [
            "well_name",
            "file_name",
            "mnemonic",
            "long_name",
            "units",
            "standard_unit",
        ]
    ]
    .drop_duplicates()
    .sort_values(
        ["mnemonic", "standard_unit", "well_name"]
    )
)

remaining_conflicts

,well_name,file_name,mnemonic,long_name,units,standard_unit
324,15_9-F-10,WLC_PETROPHYSICAL_COMPOSITE_1.DLIS,CRPM,Collar Rotational Speed,C/MI,C/MI
633,15_9-F-15,WLC_PETROPHYSICAL_COMPOSITE_2.DLIS,CRPM,Collar Rotational Speed,C/MI,C/MI
706,15_9-F-15 A,WLC_PETROPHYSICAL_COMPOSITE_2.DLIS,CRPM,Collar Rotational Speed,C/MI,C/MI
760,15_9-F-15 B,WLC_PETROPHYSICAL_COMPOSITE_1.DLIS,CRPM,Collar Rotational Speed,C/MI,C/MI
826,15_9-F-15 C,WLC_PETROPHYSICAL_COMPOSITE_1.DLIS,CRPM,Collar Rotational Speed,c/min,c/min
992,15_9-F-5,WLC_PETROPHYSICAL_COMPOSITE_1.DLIS,CRPM,Collar Rotational Speed,c/min,c/min
616,15_9-F-15,WLC_COMPOSITE_2.DLIS,DEPTH,,0.1 in,0.1 in
628,15_9-F-15,WLC_PETROPHYSICAL_COMPOSITE_2.DLIS,DEPTH,,0.1 in,0.1 in
689,15_9-F-15 A,WLC_COMPOSITE_2.DLIS,DEPTH,,0.1 in,0.1 in
701,15_9-F-15 A,WLC_PETROPHYSICAL_COMPOSITE_2.DLIS,DEPTH,,0.1 in,0.1 in


In [23]:
def classify_unit_issue(row):
    mnemonic = row["mnemonic"]
    unit = row["standard_unit"]

    # Known numerical conversion requirements
    if mnemonic == "DEPTH":
        return "conversion_required"

    if mnemonic == "TNPH":
        return "conversion_required"

    # Missing units where other records provide units
    if unit == "unknown":
        return "missing_metadata"

    # Suspected malformed weight-fraction units
    if unit == "KGF/":
        return "malformed_metadata"

    # Remaining notation differences
    if mnemonic == "CRPM":
        return "notation_standardization"

    return "no_known_issue"


catalog_df["unit_issue"] = catalog_df.apply(
    classify_unit_issue,
    axis=1
)

catalog_df["unit_issue"].value_counts()

,count
unit_issue,
no_known_issue,982
missing_metadata,40
conversion_required,38
malformed_metadata,12
notation_standardization,6


We're separating four fundamentally different data-management problems:

Unit issue

notation_standardization
│     C/MI → c/min

conversion_required
│     DEPTH: 0.1 in ↔ mm
│     TNPH: PU ↔ v/v

missing_metadata
│     blank → investigate/infer

malformed_metadata
      KGF/ → investigate


Exceptions Table

In [24]:
exceptions_df = (
    catalog_df[
        catalog_df["unit_issue"] != "no_known_issue"
    ][
        [
            "well_name",
            "file_name",
            "mnemonic",
            "long_name",
            "units",
            "standard_unit",
            "unit_issue",
        ]
    ]
    .sort_values(
        ["unit_issue", "mnemonic", "well_name"]
    )
    .reset_index(drop=True)
)

print(f"Total exceptions: {len(exceptions_df):,}")

exceptions_df.head(30)

Total exceptions: 96


,well_name,file_name,mnemonic,long_name,units,standard_unit,unit_issue
0,15_9-F-1,WLC_COMPOSITE_1.DLIS,DEPTH,,mm,mm,conversion_required
1,15_9-F-1,WLC_PETROPHYSICAL_COMPOSITE_1.DLIS,DEPTH,,mm,mm,conversion_required
2,15_9-F-1,WLC_PETROPHYSICAL_COMPOSITE_1.DLIS,DEPTH,,mm,mm,conversion_required
3,15_9-F-1 A,WLC_COMPOSITE_1.DLIS,DEPTH,,mm,mm,conversion_required
4,15_9-F-1 A,WLC_PETROPHYSICAL_COMPOSITE_1.DLIS,DEPTH,,mm,mm,conversion_required
5,15_9-F-1 B,WLC_COMPOSITE_1.DLIS,DEPTH,,mm,mm,conversion_required
6,15_9-F-1 B,WLC_PETROPHYSICAL_COMPOSITE_1.DLIS,DEPTH,,mm,mm,conversion_required
7,15_9-F-1 C,WLC_COMPOSITE_1.DLIS,DEPTH,,mm,mm,conversion_required
8,15_9-F-1 C,WLC_PETROPHYSICAL_COMPOSITE_1.DLIS,DEPTH,,mm,mm,conversion_required
9,15_9-F-11,WLC_COMPOSITE_1.DLIS,DEPTH,,mm,mm,conversion_required


Total exceptions: 96

In [25]:
exception_summary = (
    exceptions_df.groupby("unit_issue")
    .agg(
        records=("mnemonic", "count"),
        unique_mnemonics=("mnemonic", "nunique"),
        affected_wells=("well_name", "nunique"),
    )
    .sort_values("records", ascending=False)
)

exception_summary

,records,unique_mnemonics,affected_wells
unit_issue,,,
missing_metadata,40,27,7
conversion_required,38,2,17
malformed_metadata,12,12,1
notation_standardization,6,1,6


This is more useful than the raw counts because it tells us whether an issue is concentrated in one mnemonic or spread across many wells and curves.

Next, identify exactly which mnemonics belong to each issue type:

In [26]:
issue_mnemonics = (
    exceptions_df.groupby("unit_issue")["mnemonic"]
    .apply(lambda x: sorted(x.unique()))
)

for issue, mnemonics in issue_mnemonics.items():
    print(f"\n{issue.upper()}")
    print(mnemonics)


CONVERSION_REQUIRED
['DEPTH', 'TNPH']

MALFORMED_METADATA
['DWAL_WAL', 'DWCA_WAL', 'DWFE_WAL', 'DWSI_WAL', 'DWSU_WAL', 'DWTI_WAL', 'DXFE_WAL', 'WCAR', 'WCLA', 'WPYR', 'WQFM', 'WSID']

MISSING_METADATA
['APRA01M', 'APRA02M', 'APRA03M', 'APRA04M', 'APRA05M', 'APRA06M', 'APRA07M', 'APRA08M', 'APRA09M', 'APRA10M', 'APRA11M', 'APRA12M', 'APRA13M', 'APRA14M', 'APRA15M', 'APRA16M', 'AZRIT1T2', 'AZRTBM', 'BDSIM', 'CHRP', 'CHTP', 'GRSIM', 'NPCKLFM', 'SVC', 'SVHM', 'SVS', 'VPVS']

NOTATION_STANDARDIZATION
['CRPM']


At this point, Milestone 3 has evolved into:

34 DLIS files
      ↓

1,078 channel metadata records
      ↓

229 unique mnemonics
      ↓

Unit canonicalization
      ↓

96 metadata/unit exceptions
      ↓
      
Classified remediation queue

Final technical step — export the Milestone 3 outputs

In [27]:
from pathlib import Path

OUTPUT_DIR = Path("/content/outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Output directory: {OUTPUT_DIR}")

Output directory: /content/outputs


In [28]:
catalog_output = OUTPUT_DIR / "dlis_curve_catalog.csv"

catalog_df.to_csv(
    catalog_output,
    index=False
)

print(f"Catalog exported: {catalog_output}")
print(f"Rows exported: {len(catalog_df):,}")

Catalog exported: /content/outputs/dlis_curve_catalog.csv
Rows exported: 1,078


In [29]:
exceptions_output = OUTPUT_DIR / "dlis_metadata_exceptions.csv"

exceptions_df.to_csv(
    exceptions_output,
    index=False
)

print(f"Exceptions exported: {exceptions_output}")
print(f"Rows exported: {len(exceptions_df):,}")

Exceptions exported: /content/outputs/dlis_metadata_exceptions.csv
Rows exported: 96


dlis_curve_catalog.csv is your master metadata inventory. It tells us what exists in the archive.

dlis_metadata_exceptions.csv is the remediation queue. It tells us what requires additional action.

That distinction becomes particularly useful when we eventually move this workflow into SQL:

One important thing we're deliberately not doing

Don't convert TNPH, convert DEPTH, repair KGF/, or fill missing units in this notebook.

Notebook 3's job is discovery, cataloging and exception identification.

The next stage can explicitly address:

How do we transform heterogeneous source metadata and measurements into a standardized data model without destroying the original source information?

That's where we'll introduce concepts such as raw versus standardized fields, conversion rules, provenance, and validation.

## Conclusion

The Volve DLIS archive was successfully processed across 34 files representing 17 wells.

### Key Results

- 1,078 channel-level metadata records were extracted.
- 229 unique curve mnemonics were identified.
- No DLIS files failed during batch loading.
- Unit notation was standardized where differences were purely representational.
- 96 metadata and unit exceptions were identified and classified.
- 38 records require numerical unit conversion.
- 40 records contain missing unit metadata.
- 12 records contain malformed unit metadata.
- 6 records require notation standardization.

### Data-Management Insight

Metadata inconsistencies should not be treated as a single problem. The workflow separates formatting differences, numerical conversion requirements, missing metadata, and malformed metadata so that each issue can be remediated appropriately without altering source data prematurely.

### Deliverables

- `dlis_curve_catalog.csv`
- `dlis_metadata_exceptions.csv`

### Next Step

The next milestone will create standardized curve and unit mappings while preserving the original source metadata and documenting all transformations.